In [1]:
import pandas as pd
import numpy as np
import re
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import f1_score, classification_report
from sklearn.svm import LinearSVC
import torch
from transformers import AutoTokenizer, AutoModel
from torch.utils.data import Dataset, DataLoader
import torch.nn as nn
from transformers import AutoModelForSequenceClassification
from torch.optim import AdamW
from sklearn.preprocessing import LabelEncoder

## Cell 1: Imports
| `re` | Regular expressions για καθαρισμό κειμένου |

| `numpy` | Αριθμητικές πράξεις |

| `pandas` | Φόρτωση και χειρισμός CSV δεδομένων |

| `TfidfVectorizer` | Μετατροπή κειμένου σε αριθμητικά διανύσματα |

| `LogisticRegression` | Ταξινομητής για τις δύο κατηγορίες |

| `f1_score, classification_report` | Αξιολόγηση αποτελεσμάτων |

In [2]:
train = pd.read_csv('train.csv')
test = pd.read_csv('test.csv')
valid = pd.read_csv('valid.csv')

print(f'Train: {train.shape[0]} rows, {train.shape[1]} columns')
print(f'Valid: {valid.shape[0]} rows, {valid.shape[1]} columns')
print(f'Test: {test.shape[0]} rows, {test.shape[1]} columns')

print('\nTrain columns:', train.columns.tolist())

print('\n--- First 3 rows of Train ---')
train[['title', 'hazard-category', 'product-category']].head(3)


Train: 5082 rows, 11 columns
Valid: 565 rows, 11 columns
Test: 997 rows, 7 columns

Train columns: ['Unnamed: 0', 'year', 'month', 'day', 'country', 'title', 'text', 'hazard-category', 'product-category', 'hazard', 'product']

--- First 3 rows of Train ---


,title,hazard-category,product-category
0,Recall Notification: FSIS-024-94,biological,"meat, egg and dairy products"
1,Recall Notification: FSIS-033-94,biological,"meat, egg and dairy products"
2,Recall Notification: FSIS-014-94,biological,"meat, egg and dairy products"


## Cell 2: Φόρτωση Δεδομένων
- **train.csv**: 5082 δείγματα με labels — για εκπαίδευση

- **valid.csv**: 565 δείγματα με labels — για τοπική αξιολόγηση

- **test.csv**: 997 δείγματα χωρίς labels — για submission στο Kaggle

Κάθε δείγμα περιέχει: `title`, `text`, `year`, `month`, `day`, `country`

και τα labels: `hazard-category`, `product-category`.

In [3]:
print('=== Hazard Categories (10 classes) ===')
hazard_counts = train['hazard-category'].value_counts()
print(hazard_counts)
print(f'\nMax: {hazard_counts.max()} | Min: {hazard_counts.min()} | Analogy: {hazard_counts.max()//hazard_counts.min()}x')
print('\n=== Product Categories (22 classes) ===')
product_counts = train['product-category'].value_counts()
print(product_counts)
print(f'\nMax: {product_counts.max()} | Min: {product_counts.min()} | Analogy: {product_counts.max()//product_counts.min()}x')

=== Hazard Categories (10 classes) ===
hazard-category
allergens                         1854
biological                        1741
foreign bodies                     561
fraud                              371
chemical                           287
other hazard                       134
packaging defect                    54
organoleptic aspects                53
food additives and flavourings      24
migration                            3
Name: count, dtype: int64

Max: 1854 | Min: 3 | Analogy: 618x

=== Product Categories (22 classes) ===
product-category
meat, egg and dairy products                         1434
cereals and bakery products                           671
fruits and vegetables                                 535
prepared dishes and snacks                            469
seafood                                               268
soups, broths, sauces and condiments                  264
nuts, nut products and seeds                          262
ices and desserts            

## Cell 3: Ανάλυση Κατανομής Κλάσεων (EDA)

### Παρατηρήσεις:
- **Hazard**: 10 κλάσεις με αναλογία 618x (allergens: 1854 vs migration: 3)
- **Product**: 22 κλάσεις με αναλογία 286x (meat/dairy: 1434 vs sugars: 5)

### Πρόβλημα Class Imbalance:
Ένας ταξινομητής που αγνοεί τις μικρές κλάσεις έχει υψηλή
accuracy αλλά χαμηλό macro-F1. Για αυτό χρησιμοποιούμε
`class_weight='balanced'` στο Logistic Regression.

In [4]:
def preprocess_text(text):
    if not isinstance(text, str):
        return ''

    text = text.lower() # Convert to lowercase
    text = re.sub(r'\d+', ' ', text) # Remove digits
    text = re.sub(r'[^a-z\s]', ' ', text) # Remove punctuation
    text = re.sub(r'\s+', ' ', text).strip() # Remove extra whitespace
    return text

train['title_clean'] = train['title'].apply(preprocess_text)
valid['title_clean'] = valid['title'].apply(preprocess_text)
test['title_clean'] = test['title'].apply(preprocess_text)

print('--- Preprocessing Example ---')
for i in range(3):
    print(f'Original: {train["title"].iloc[i]}')
    print(f'Cleaned: {train["title_clean"].iloc[i]}\n')
    print()

--- Preprocessing Example ---
Original: Recall Notification: FSIS-024-94
Cleaned: recall notification fsis


Original: Recall Notification: FSIS-033-94
Cleaned: recall notification fsis


Original: Recall Notification: FSIS-014-94
Cleaned: recall notification fsis




## Cell 4: Preprocessing

Καθαρισμός κειμένου σε 4 βήματα:
1. **Lowercase**: ομοιομορφία κεφαλαίων/πεζών
2. **Αφαίρεση αριθμών**: δεν συνεισφέρουν στην κατηγοριοποίηση
3. **Αφαίρεση ειδικών χαρακτήρων**: κρατάμε μόνο γράμματα και κενά
4. **Αφαίρεση περιττών κενών**: ομοιομορφία spacing

Χρησιμοποιούμε το `title` ως feature γιατί είναι σύντομο
και περιεκτικό. Το `text` θα δοκιμαστεί σε επόμενη φάση.

In [5]:
# κρατάμε τις 10.000 πιο σημαντικές λέξεις/φράσεις
# unigrams και bigrams
# χρησιμοποιούμε log(tf) αντί για tf για να μειώσουμε την επίδραση των πολύ συχνών λέξεων
tfidf = TfidfVectorizer(max_features=10000, ngram_range=(1,2), sublinear_tf=True)

# fit_transform στο train: μαθαίνει το λεξιλόγιο και μετατρέπει
X_train = tfidf.fit_transform(train['title_clean'])

# transform μόνο σε valid/test: χρησιμοποιεί το λεξιλόγιο του train
X_valid = tfidf.transform(valid['title_clean'])
X_test = tfidf.transform(test['title_clean'])

print(f'X_train shape: {X_train.shape}')
print(f'X_valid shape: {X_valid.shape}')
print(f'X_test shape: {X_test.shape}')
print(f'\nVocabulary: {len(tfidf.vocabulary_)} words/phrases')

y_train_hazard = train['hazard-category']
y_train_product = train['product-category']
y_valid_hazard = valid['hazard-category']
y_valid_product = valid['product-category']

print('\n Features and labels prepared')

X_train shape: (5082, 10000)
X_valid shape: (565, 10000)
X_test shape: (997, 10000)

Vocabulary: 10000 words/phrases

 Features and labels prepared


## Cell 5: TF-IDF Feature Extraction

Μετατροπή κειμένου σε αριθμητικά διανύσματα με TF-IDF.

**Παράμετροι:**
- `max_features=10000`: οι 10.000 πιο σημαντικές λέξεις/φράσεις
- `ngram_range=(1,2)`: unigrams και bigrams
- `sublinear_tf=True`: log normalization για την TF

Το `fit_transform` γίνεται ΜΟΝΟ στο train set. Στο valid και test
κάνουμε μόνο `transform`, ώστε να μην διαρρεύσουν πληροφορίες
από τα test δεδομένα στην εκπαίδευση.

In [6]:
#Classifier for hazard category
clf_hazard = LogisticRegression(C=1.0, class_weight='balanced', max_iter=1000, random_state=42)

clf_hazard.fit(X_train, y_train_hazard)
print('Hazard classifier trained')

#Classifier for product-category
clf_product = LogisticRegression(C=1.0, class_weight='balanced', max_iter=1000, random_state=42)

clf_product.fit(X_train, y_train_product)
print('Product classifier trained')

Hazard classifier trained
Product classifier trained


## Cell 6: Εκπαίδευση Logistic Regression

Εκπαιδεύουμε **δύο ξεχωριστούς classifiers**:
- Έναν για το `hazard-category` (10 κλάσεις)
- Έναν για το `product-category` (22 κλάσεις)

**Παράμετροι:**
- `C=1.0`: regularization για αποφυγή overfitting
- `class_weight='balanced'`: μεγαλύτερο βάρος στις σπάνιες κλάσεις
- `max_iter=1000`: μέγιστες επαναλήψεις για σύγκλιση
- `random_state=42`: για reproducibility των αποτελεσμάτων

In [7]:
def official_st1_score(y_true_hazard, y_pred_hazard, y_true_product, y_pred_product, verbose=True):
    #Βήμα 1: macro-F1 για hazard σε όλα τα δείγματα
    f1_hazard = f1_score(y_true_hazard, y_pred_hazard, average='macro', zero_division=0)

    # Βήμα 2: βρίσκουμε τα δείγματα όπου το hazard ήταν σωστό
    y_true_hazard = np.array(y_true_hazard)
    y_pred_hazard = np.array(y_pred_hazard)
    y_true_product = np.array(y_true_product)
    y_pred_product = np.array(y_pred_product)

    correct_hazard_mask = (y_true_hazard == y_pred_hazard)
    n_correct = correct_hazard_mask.sum()

    # Βήμα 3: macro-F1 για product ΜΟΝΟ στα σωστά hazard
    if n_correct ==0:
        f1_product = 0.0
    else:
        f1_product = f1_score(y_true_product[correct_hazard_mask], y_pred_product[correct_hazard_mask], average='macro', zero_division=0)


    # Βήμα 4: official score
    score = (f1_hazard + f1_product) / 2

    if verbose:
        print(f'macro-F1 Hazard:   {f1_hazard:.4f}')
        print(f'Correct hazard predictions:  {n_correct}/{len(y_true_hazard)} ({100*n_correct/len(y_true_hazard):.1f}%)')
        print(f'macro-F1 Product (given correct h): {f1_product:.4f}')
        print(f'----------------')
        print(f'OFFICIAL ST1 SCORE:   {score:.4f}')
    return score

# Predictions στο validation set
pred_hazard_valid = clf_hazard.predict(X_valid)
pred_product_valid = clf_product.predict(X_valid)

print('=== VALIDATION SET ===\n')
score = official_st1_score(
    valid['hazard-category'], pred_hazard_valid, valid['product-category'], pred_product_valid)

               

=== VALIDATION SET ===

macro-F1 Hazard:   0.6471
Correct hazard predictions:  456/565 (80.7%)
macro-F1 Product (given correct h): 0.5723
----------------
OFFICIAL ST1 SCORE:   0.6097


## Cell 7: Αξιολόγηση — Official ST1 Metric

**Αποτελέσματα Validation Set:**
| Metric | Τιμή |
|---|---|
| macro-F1 Hazard | 0.6471 |
| Σωστά Hazard Predictions | 456/565 (80.7%) |
| macro-F1 Product (given correct hazard) | 0.5723 |
| **Official ST1 Score** | **0.6097** |

Το baseline με TF-IDF + Logistic Regression επιτυγχάνει
score 0.6097. Το hazard classifier έχει καλύτερη απόδοση
από το product, κάτι αναμενόμενο λόγω του μικρότερου
αριθμού κλάσεων (10 vs 22).

In [8]:
print('=== Hazard Category - Classification Report ===\n')
print(classification_report(valid['hazard-category'], pred_hazard_valid, zero_division=0))

print('\n=== Product Category - Classification Report ===\n')
print(classification_report(valid['product-category'], pred_product_valid, zero_division=0))

=== Hazard Category - Classification Report ===

                                precision    recall  f1-score   support

                     allergens       0.89      0.80      0.84       207
                    biological       0.92      0.90      0.91       194
                      chemical       0.62      0.71      0.67        28
food additives and flavourings       0.50      0.50      0.50         2
                foreign bodies       0.71      0.81      0.76        63
                         fraud       0.62      0.63      0.63        41
          organoleptic aspects       0.50      0.75      0.60         8
                  other hazard       0.32      0.43      0.36        14
              packaging defect       0.50      0.62      0.56         8

                      accuracy                           0.81       565
                     macro avg       0.62      0.69      0.65       565
                  weighted avg       0.82      0.81      0.81       565


=== Product

## Cell 8: Αναλυτική Αξιολόγηση ανά Κλάση

### Hazard Category:
- Καλή απόδοση στις μεγάλες κλάσεις: `biological` (F1=0.91), `allergens` (F1=0.84)
- Χαμηλή απόδοση στις μικρές κλάσεις: `other hazard` (F1=0.36)
- Η κλάση `migration` (3 δείγματα) δεν προβλέφθηκε καθόλου

### Product Category:
- macro avg F1: 0.58 — χαμηλότερο λόγω των 22 κλάσεων
- Το class imbalance επηρεάζει σημαντικά τις μικρές κλάσεις

### Συμπέρασμα:
Το μοντέλο δυσκολεύεται με τις σπάνιες κλάσεις παρά το
`class_weight='balanced'`. Απαιτείται καλύτερη στρατηγική
για το class imbalance στις επόμενες φάσεις.

In [9]:
#Predictions on test set
pred_hazard_test = clf_hazard.predict(X_test)
pred_product_test = clf_product.predict(X_test)

submission = pd.DataFrame({
    'id': test['id'],
    'hazard-category': pred_hazard_test,
    'product-category': pred_product_test
})

#save to csv
submission.to_csv('submission_phase1_logreg.csv', index=False)

print('Submission file saved: submission_phase1_logreg.csv')
print(f'Number of predictions: {len(submission)}')
print('\n--- First 5 predictions ---')
print(submission.head())

Submission file saved: submission_phase1_logreg.csv
Number of predictions: 997

--- First 5 predictions ---
   id       hazard-category              product-category
0   0            biological  meat, egg and dairy products
1   1            biological  meat, egg and dairy products
2   2            biological  meat, egg and dairy products
3   3            biological                       seafood
4   4  organoleptic aspects             ices and desserts


## Cell 9: Παραγωγή Submission File

Εκτελούμε τους εκπαιδευμένους classifiers στο test set
και αποθηκεύουμε τις προβλέψεις σε CSV για submission στο Kaggle.

**Αρχείο:** `submission_phase1_logreg.csv`
**Predictions:** 997 (ένα για κάθε δείγμα του test set)
**Στήλες:** `id`, `hazard-category`, `product-category`

In [10]:
print('=== ΣΥΝΟΨΗ ΦΑΣΗΣ 1 ===')
print()
print('ΔΕΔΟΜΕΝΑ:')
print(f'  Train:      {train.shape[0]} δείγματα')
print(f'  Validation: {valid.shape[0]} δείγματα')
print(f'  Test:       {test.shape[0]} δείγματα')
print()
print('FEATURES:')
print(f'  Input:      title (καθαρισμένο)')
print(f'  Μέθοδος:    TF-IDF (max_features=10000, ngram_range=(1,2))')
print()
print('ΜΟΝΤΕΛΟ:')
print(f'  Classifier: Logistic Regression')
print(f'  C:          1.0')
print(f'  Weights:    balanced')
print()
print('ΑΠΟΤΕΛΕΣΜΑΤΑ VALIDATION:')
print(f'  macro-F1 Hazard:   0.6471')
print(f'  macro-F1 Product:  0.5723')
print(f'  Official Score:    0.6097')
print()
print('ΕΠΟΜΕΝΑ ΒΗΜΑΤΑ:')
print('  1. Δοκιμή SVM classifier')
print('  2. Προσθήκη stop words / stemming στο preprocessing')
print('  3. Συνδυασμός title + text ως features')
print('  4. Neural baseline με BERT (Φάση 2)')

=== ΣΥΝΟΨΗ ΦΑΣΗΣ 1 ===

ΔΕΔΟΜΕΝΑ:
  Train:      5082 δείγματα
  Validation: 565 δείγματα
  Test:       997 δείγματα

FEATURES:
  Input:      title (καθαρισμένο)
  Μέθοδος:    TF-IDF (max_features=10000, ngram_range=(1,2))

ΜΟΝΤΕΛΟ:
  Classifier: Logistic Regression
  C:          1.0
  Weights:    balanced

ΑΠΟΤΕΛΕΣΜΑΤΑ VALIDATION:
  macro-F1 Hazard:   0.6471
  macro-F1 Product:  0.5723
  Official Score:    0.6097

ΕΠΟΜΕΝΑ ΒΗΜΑΤΑ:
  1. Δοκιμή SVM classifier
  2. Προσθήκη stop words / stemming στο preprocessing
  3. Συνδυασμός title + text ως features
  4. Neural baseline με BERT (Φάση 2)


## Cell 10: Σύνοψη Φάσης 1

| Βήμα | Τι κάναμε | Γιατί |
|---|---|---|
| Preprocessing | lowercase, αφαίρεση αριθμών/ειδικών χαρακτήρων | Ομοιομορφία κειμένου |
| Features | TF-IDF (unigrams + bigrams, 10k features) | Αριθμητική αναπαράσταση κειμένου |
| Classifier | Logistic Regression | Γρήγορο, ερμηνεύσιμο, καλό για text |
| Imbalance | `class_weight='balanced'` | Αντισταθμίζει τις σπάνιες κλάσεις |
| Metric | Official ST1 score | Συνεπές με την αξιολόγηση του Kaggle |

**Αποτέλεσμα: Official ST1 Score = 0.6097**

### Tool Usage Statement
Ο κώδικας αυτός δημιουργήθηκε με βοήθεια Claude για:
- Εξήγηση εννοιών (TF-IDF, macro-F1, class imbalance)
- Δημιουργία και debugging κώδικα
- Υλοποίηση του official ST1 metric

In [11]:
#Combine title and text
train['combined'] = train['title'].apply(preprocess_text) + '' + train['text'].apply(preprocess_text)
valid['combined'] = valid['title'].apply(preprocess_text) + '' + train['text'].apply(preprocess_text)
test['combined'] = test['title'].apply(preprocess_text) + '' + train['text'].apply(preprocess_text)

#New TF-IDF
tfidf_combined = TfidfVectorizer(
    max_features=10000,
    ngram_range=(1, 2),
    sublinear_tf=True,
    stop_words='english'
) 

X_train_combined = tfidf_combined.fit_transform(train['combined'])
X_valid_combined = tfidf_combined.transform(valid['combined'])
X_test_combined = tfidf_combined.transform(test['combined'])

print(f'X_train shape: {X_train_combined.shape}')
print(f'X_valid shape: {X_valid_combined.shape}')
print(f'X_test shape: {X_test_combined.shape}')


X_train shape: (5082, 10000)
X_valid shape: (565, 10000)
X_test shape: (997, 10000)


In [12]:
#train new classifier with combined features
clf_hazard_combined = LogisticRegression(
    C=1.0,
    class_weight='balanced',
    max_iter=1000,
    random_state=42
)

clf_hazard_combined.fit(X_train_combined, y_train_hazard)
print('Hazard classifier trained')

clf_product_combined = LogisticRegression(
    C=1.0,
    class_weight='balanced',
    max_iter=1000,
    random_state=42
)

clf_product_combined.fit(X_train_combined, y_train_product)
print('Product classifier trained')

pred_hazard_combined = clf_hazard_combined.predict(X_valid_combined)
pred_product_combined = clf_product_combined.predict(X_valid_combined)

print('\n=== Rate(title + text) ===\n')
score_combined = official_st1_score(
    valid['hazard-category'], pred_hazard_combined,
    valid['product-category'], pred_product_combined
)

print('\n=== Comparisson ===')
print(f'Baseline (title only): 0.6097')
print(f'Combined (title+text): {score_combined:.4f}')

Hazard classifier trained
Product classifier trained

=== Rate(title + text) ===

macro-F1 Hazard:   0.1306
Correct hazard predictions:  149/565 (26.4%)
macro-F1 Product (given correct h): 0.1104
----------------
OFFICIAL ST1 SCORE:   0.1205

=== Comparisson ===
Baseline (title only): 0.6097
Combined (title+text): 0.1205


## Βελτίωση 1: Προσθήκη Text ως Feature (Πρώτη Δοκιμή)

Συνδυασμός `title` + `text` με TF-IDF και αφαίρεση stop words.

**Αποτέλεσμα: 0.1205 — χειρότερο από το baseline (0.6097)**

Το πλήρες `text` περιέχει πάρα πολύ "θόρυβο" (διευθύνσεις,
ημερομηνίες, αριθμούς) που μπερδεύει το μοντέλο.
**Επόμενη δοκιμή:** περιορισμός στα πρώτα 200 χαρακτήρες του text.

In [13]:
# Κρατάμε μόνο τους πρώτους 200 χαρακτήρες του text
train['combined'] = train['title'].apply(preprocess_text) + ' ' + train['text'].str[:200].apply(preprocess_text)
valid['combined'] = valid['title'].apply(preprocess_text) + ' ' + valid['text'].str[:200].apply(preprocess_text)
test['combined']  = test['title'].apply(preprocess_text) + ' ' + test['text'].str[:200].apply(preprocess_text)

tfidf_combined2 = TfidfVectorizer(
    max_features=10000,
    ngram_range=(1, 2),
    sublinear_tf=True,
    stop_words='english'
)

X_train_combined2 = tfidf_combined2.fit_transform(train['combined'])
X_valid_combined2 = tfidf_combined2.transform(valid['combined'])
X_test_combined2  = tfidf_combined2.transform(test['combined'])

clf_hazard_combined2 = LogisticRegression(
    C=1.0,
    class_weight='balanced',
    max_iter=1000,
    random_state=42
)
clf_hazard_combined2.fit(X_train_combined2, y_train_hazard)

clf_product_combined2 = LogisticRegression(
    C=1.0,
    class_weight='balanced',
    max_iter=1000,
    random_state=42
)
clf_product_combined2.fit(X_train_combined2, y_train_product)

pred_hazard_combined2  = clf_hazard_combined2.predict(X_valid_combined2)
pred_product_combined2 = clf_product_combined2.predict(X_valid_combined2)

print('=== ΑΞΙΟΛΟΓΗΣΗ (title + text[:200]) ===\n')
score_combined2 = official_st1_score(
    valid['hazard-category'], pred_hazard_combined2,
    valid['product-category'], pred_product_combined2
)

print(f'\n=== ΣΥΓΚΡΙΣΗ ===')
print(f'Baseline (title only):      0.6097')
print(f'Combined (title+full text): 0.1205')
print(f'Combined (title+text 200):  {score_combined2:.4f}')

=== ΑΞΙΟΛΟΓΗΣΗ (title + text[:200]) ===

macro-F1 Hazard:   0.6927
Correct hazard predictions:  470/565 (83.2%)
macro-F1 Product (given correct h): 0.5887
----------------
OFFICIAL ST1 SCORE:   0.6407

=== ΣΥΓΚΡΙΣΗ ===
Baseline (title only):      0.6097
Combined (title+full text): 0.1205
Combined (title+text 200):  0.6407


## Βελτίωση 1β: Προσθήκη Text[:200] ως Feature

Δοκιμή με τους πρώτους 200 χαρακτήρες του `text` αντί για το πλήρες.

**Αποτελέσματα:**
| Μέθοδος | Score |
|---|---|
| Baseline (title only) | 0.6097 |
| Combined (title + full text) | 0.1205 |
| Combined (title + text[:200]) | **0.6407** |

**Συμπέρασμα:** Οι πρώτοι 200 χαρακτήρες του text περιέχουν
τις πιο χρήσιμες πληροφορίες. Το πλήρες text εισάγει πάρα
πολύ θόρυβο που βλάπτει την απόδοση.

In [14]:
# Δοκιμή με text[:500]
train['combined'] = train['title'].apply(preprocess_text) + ' ' + train['text'].str[:500].apply(preprocess_text)
valid['combined'] = valid['title'].apply(preprocess_text) + ' ' + valid['text'].str[:500].apply(preprocess_text)
test['combined']  = test['title'].apply(preprocess_text) + ' ' + test['text'].str[:500].apply(preprocess_text)

tfidf_combined3 = TfidfVectorizer(
    max_features=10000,
    ngram_range=(1, 2),
    sublinear_tf=True,
    stop_words='english'
)

X_train_combined3 = tfidf_combined3.fit_transform(train['combined'])
X_valid_combined3 = tfidf_combined3.transform(valid['combined'])
X_test_combined3  = tfidf_combined3.transform(test['combined'])

clf_hazard_combined3 = LogisticRegression(
    C=1.0,
    class_weight='balanced',
    max_iter=1000,
    random_state=42
)
clf_hazard_combined3.fit(X_train_combined3, y_train_hazard)

clf_product_combined3 = LogisticRegression(
    C=1.0,
    class_weight='balanced',
    max_iter=1000,
    random_state=42
)
clf_product_combined3.fit(X_train_combined3, y_train_product)

pred_hazard_combined3  = clf_hazard_combined3.predict(X_valid_combined3)
pred_product_combined3 = clf_product_combined3.predict(X_valid_combined3)

print('=== ΑΞΙΟΛΟΓΗΣΗ (title + text[:500]) ===\n')
score_combined3 = official_st1_score(
    valid['hazard-category'], pred_hazard_combined3,
    valid['product-category'], pred_product_combined3
)

print(f'\n=== ΣΥΓΚΡΙΣΗ ===')
print(f'Baseline (title only):      0.6097')
print(f'Combined (title+full text): 0.1205')
print(f'Combined (title+text 200):  0.6407')
print(f'Combined (title+text 500):  {score_combined3:.4f}')

=== ΑΞΙΟΛΟΓΗΣΗ (title + text[:500]) ===

macro-F1 Hazard:   0.7593
Correct hazard predictions:  508/565 (89.9%)
macro-F1 Product (given correct h): 0.5725
----------------
OFFICIAL ST1 SCORE:   0.6659

=== ΣΥΓΚΡΙΣΗ ===
Baseline (title only):      0.6097
Combined (title+full text): 0.1205
Combined (title+text 200):  0.6407
Combined (title+text 500):  0.6659


In [15]:
# Δοκιμή με text[:1000]
train['combined'] = train['title'].apply(preprocess_text) + ' ' + train['text'].str[:1000].apply(preprocess_text)
valid['combined'] = valid['title'].apply(preprocess_text) + ' ' + valid['text'].str[:1000].apply(preprocess_text)
test['combined']  = test['title'].apply(preprocess_text) + ' ' + test['text'].str[:1000].apply(preprocess_text)

tfidf_combined4 = TfidfVectorizer(
    max_features=10000,
    ngram_range=(1, 2),
    sublinear_tf=True,
    stop_words='english'
)

X_train_combined4 = tfidf_combined4.fit_transform(train['combined'])
X_valid_combined4 = tfidf_combined4.transform(valid['combined'])
X_test_combined4  = tfidf_combined4.transform(test['combined'])

clf_hazard_combined4 = LogisticRegression(
    C=1.0,
    class_weight='balanced',
    max_iter=1000,
    random_state=42
)
clf_hazard_combined4.fit(X_train_combined4, y_train_hazard)

clf_product_combined4 = LogisticRegression(
    C=1.0,
    class_weight='balanced',
    max_iter=1000,
    random_state=42
)
clf_product_combined4.fit(X_train_combined4, y_train_product)

pred_hazard_combined4  = clf_hazard_combined4.predict(X_valid_combined4)
pred_product_combined4 = clf_product_combined4.predict(X_valid_combined4)

print('=== ΑΞΙΟΛΟΓΗΣΗ (title + text[:1000]) ===\n')
score_combined4 = official_st1_score(
    valid['hazard-category'], pred_hazard_combined4,
    valid['product-category'], pred_product_combined4
)

print(f'\n=== ΣΥΓΚΡΙΣΗ ===')
print(f'Baseline (title only):      0.6097')
print(f'Combined (title+full text): 0.1205')
print(f'Combined (title+text 200):  0.6407')
print(f'Combined (title+text 500):  0.6659')
print(f'Combined (title+text 1000): {score_combined4:.4f}')

=== ΑΞΙΟΛΟΓΗΣΗ (title + text[:1000]) ===

macro-F1 Hazard:   0.7589
Correct hazard predictions:  517/565 (91.5%)
macro-F1 Product (given correct h): 0.5408
----------------
OFFICIAL ST1 SCORE:   0.6499

=== ΣΥΓΚΡΙΣΗ ===
Baseline (title only):      0.6097
Combined (title+full text): 0.1205
Combined (title+text 200):  0.6407
Combined (title+text 500):  0.6659
Combined (title+text 1000): 0.6499


In [16]:
# Δοκιμή με text[:550]
train['combined'] = train['title'].apply(preprocess_text) + ' ' + train['text'].str[:550].apply(preprocess_text)
valid['combined'] = valid['title'].apply(preprocess_text) + ' ' + valid['text'].str[:550].apply(preprocess_text)
test['combined']  = test['title'].apply(preprocess_text) + ' ' + test['text'].str[:550].apply(preprocess_text)

tfidf_combined4 = TfidfVectorizer(
    max_features=10000,
    ngram_range=(1, 2),
    sublinear_tf=True,
    stop_words='english'
)

X_train_combined4 = tfidf_combined4.fit_transform(train['combined'])
X_valid_combined4 = tfidf_combined4.transform(valid['combined'])
X_test_combined4  = tfidf_combined4.transform(test['combined'])

clf_hazard_combined4 = LogisticRegression(
    C=1.0,
    class_weight='balanced',
    max_iter=1000,
    random_state=42
)
clf_hazard_combined4.fit(X_train_combined4, y_train_hazard)

clf_product_combined4 = LogisticRegression(
    C=1.0,
    class_weight='balanced',
    max_iter=1000,
    random_state=42
)
clf_product_combined4.fit(X_train_combined4, y_train_product)

pred_hazard_combined4  = clf_hazard_combined4.predict(X_valid_combined4)
pred_product_combined4 = clf_product_combined4.predict(X_valid_combined4)

print('=== ΑΞΙΟΛΟΓΗΣΗ (title + text[:1000]) ===\n')
score_combined4 = official_st1_score(
    valid['hazard-category'], pred_hazard_combined4,
    valid['product-category'], pred_product_combined4
)

print(f'\n=== ΣΥΓΚΡΙΣΗ ===')
print(f'Baseline (title only):      0.6097')
print(f'Combined (title+full text): 0.1205')
print(f'Combined (title+text 200):  0.6407')
print(f'Combined (title+text 500):  0.6659')
print(f'Combined (title+text 1000): 0.6499')
print(f'Combined (title+text 550): {score_combined4:.4f}')

=== ΑΞΙΟΛΟΓΗΣΗ (title + text[:1000]) ===

macro-F1 Hazard:   0.7726
Correct hazard predictions:  511/565 (90.4%)
macro-F1 Product (given correct h): 0.5697
----------------
OFFICIAL ST1 SCORE:   0.6712

=== ΣΥΓΚΡΙΣΗ ===
Baseline (title only):      0.6097
Combined (title+full text): 0.1205
Combined (title+text 200):  0.6407
Combined (title+text 500):  0.6659
Combined (title+text 1000): 0.6499
Combined (title+text 550): 0.6712


## Βελτίωση 1γ: Εύρεση Βέλτιστου Μεγέθους Text

Δοκιμή διαφορετικών μεγεθών text για εύρεση του βέλτιστου.

| Μέθοδος | Score |
|---|---|
| Baseline (title only) | 0.6097 |
| Combined (title + full text) | 0.1205 |
| Combined (title + text[:200]) | 0.6407 |
| Combined (title + text[:500]) | 0.6659 |
| Combined (title + text[:1000]) | 0.6499 |
| Combined (title + text[:550]) | **0.6712** |

**Συμπέρασμα:** Το βέλτιστο μέγεθος είναι text[:550].
Μετά τους 550 χαρακτήρες εισάγεται θόρυβος που μειώνει
την απόδοση. Η ποιότητα του input είναι πιο σημαντική
από την ποσότητα.

In [17]:
# Submission με το βέλτιστο μοντέλο (title + text[:550])
train['combined'] = train['title'].apply(preprocess_text) + ' ' + train['text'].str[:550].apply(preprocess_text)
valid['combined'] = valid['title'].apply(preprocess_text) + ' ' + valid['text'].str[:550].apply(preprocess_text)
test['combined']  = test['title'].apply(preprocess_text) + ' ' + test['text'].str[:550].apply(preprocess_text)

tfidf_best = TfidfVectorizer(
    max_features=10000,
    ngram_range=(1, 2),
    sublinear_tf=True,
    stop_words='english'
)

X_train_best = tfidf_best.fit_transform(train['combined'])
X_valid_best = tfidf_best.transform(valid['combined'])
X_test_best  = tfidf_best.transform(test['combined'])

clf_hazard_best = LogisticRegression(
    C=1.0,
    class_weight='balanced',
    max_iter=1000,
    random_state=42
)
clf_hazard_best.fit(X_train_best, y_train_hazard)

clf_product_best = LogisticRegression(
    C=1.0,
    class_weight='balanced',
    max_iter=1000,
    random_state=42
)
clf_product_best.fit(X_train_best, y_train_product)

# Επαλήθευση στο validation
pred_hazard_best  = clf_hazard_best.predict(X_valid_best)
pred_product_best = clf_product_best.predict(X_valid_best)

print('=== ΤΕΛΙΚΗ ΑΞΙΟΛΟΓΗΣΗ (title + text[:550]) ===\n')
official_st1_score(
    valid['hazard-category'], pred_hazard_best,
    valid['product-category'], pred_product_best
)

# Παραγωγή submission
pred_hazard_test  = clf_hazard_best.predict(X_test_best)
pred_product_test = clf_product_best.predict(X_test_best)

submission = pd.DataFrame({
    'id': test['id'],
    'hazard-category': pred_hazard_test,
    'product-category': pred_product_test
})

submission.to_csv('submission_phase1_best.csv', index=False)
print('\nSubmission file saved: submission_phase1_best.csv')
print(f'Αριθμός predictions: {len(submission)}')
print('\n--- Πρώτες 5 προβλέψεις ---')
print(submission.head())

=== ΤΕΛΙΚΗ ΑΞΙΟΛΟΓΗΣΗ (title + text[:550]) ===

macro-F1 Hazard:   0.7726
Correct hazard predictions:  511/565 (90.4%)
macro-F1 Product (given correct h): 0.5697
----------------
OFFICIAL ST1 SCORE:   0.6712

Submission file saved: submission_phase1_best.csv
Αριθμός predictions: 997

--- Πρώτες 5 προβλέψεις ---
   id hazard-category              product-category
0   0      biological  meat, egg and dairy products
1   1      biological  meat, egg and dairy products
2   2      biological  meat, egg and dairy products
3   3      biological                       seafood
4   4  foreign bodies             ices and desserts


## Τελικό Submission Φάσης 1

Εκπαίδευση με το βέλτιστο συνδυασμό: `title + text[:550]`

**Τελικά Αποτελέσματα Validation:**
| Metric | Baseline | Βέλτιστο |
|---|---|---|
| macro-F1 Hazard | 0.6471 | 0.7726 |
| Σωστά Hazard | 80.7% | 90.4% |
| macro-F1 Product | 0.5723 | 0.5697 |
| **Official Score** | **0.6097** | **0.6712** |

**Αρχείο:** `submission_phase1_best.csv`

In [18]:
#SVM for hazard
clf_svm_hazard = LinearSVC(
    C=1.0,
    class_weight='balanced',
    max_iter = 2000,
    random_state=42
)

clf_svm_hazard.fit(X_train_best, y_train_hazard)
print('SVM hazard trianed')

#SVM for product
clf_svm_product = LinearSVC(
    C=1.0,
    class_weight='balanced',
    max_iter=2000,
    random_state=42
)

clf_svm_product.fit(X_train_best, y_train_product)
print('SVM Porduct trained')

pred_svm_hazard = clf_svm_hazard.predict(X_valid_best)
pred_svm_product = clf_svm_product.predict(X_valid_best)

print('\n=== Rating SVM (title + text[:550]) ===\n')
score_svm = official_st1_score(
    valid['hazard-category'], pred_svm_hazard,
    valid['product-category'], pred_svm_product
)

print(f'\n=== Comparisson ===')
print(f'Logistic Regression (title+text[:550]): 0.6712')
print(f' SVM                ( title+text[:550]): {score_svm:.4f}')

SVM hazard trianed
SVM Porduct trained

=== Rating SVM (title + text[:550]) ===

macro-F1 Hazard:   0.7904
Correct hazard predictions:  525/565 (92.9%)
macro-F1 Product (given correct h): 0.6622
----------------
OFFICIAL ST1 SCORE:   0.7263

=== Comparisson ===
Logistic Regression (title+text[:550]): 0.6712
 SVM                ( title+text[:550]): 0.7263


## Βελτίωση 2: SVM Classifier

Αντικατάσταση Logistic Regression με LinearSVC.

**Αποτελέσματα:**
| Classifier | Score |
|---|---|
| Logistic Regression (title+text[:550]) | 0.6712 |
| **LinearSVC (title+text[:550])** | **0.7263** |

**Παράμετροι:**
- `C=1.0`: regularization
- `class_weight='balanced'`: αντιστάθμιση class imbalance
- `max_iter=2000`: επαρκείς επαναλήψεις για σύγκλιση

In [19]:
#Predictions on test set with SVM
pred_svm_hazard_test = clf_svm_hazard.predict(X_test_best)
pred_svm_product_test = clf_svm_product.predict(X_test_best)

submission_svm = pd.DataFrame({
    'id': test['id'],
    'hazard-category': pred_svm_hazard_test,
    'product-category': pred_svm_product_test
})

submission_svm.to_csv('submission_phase1_svm.csv', index=False)

print('Submission file saved: submission_phase1_svm.csv')
print(f'Number of predictions: {len(submission_svm)}')
print('\n--- First 5 predictions ---')
print(submission_svm.head())

Submission file saved: submission_phase1_svm.csv
Number of predictions: 997

--- First 5 predictions ---
   id hazard-category              product-category
0   0      biological  meat, egg and dairy products
1   1      biological  meat, egg and dairy products
2   2      biological  meat, egg and dairy products
3   3      biological                       seafood
4   4  foreign bodies             ices and desserts


## Submission 3: SVM (title + text[:550])

**Kaggle Score: 0.72822**

| Submission | Kaggle Score |
|---|---|
| Baseline (title only, LogReg) | 0.57609 |
| LogReg (title + text[:550]) | 0.65231 |
| **SVM (title + text[:550])** | **0.72822** |

Συνολική βελτίωση +0.152 με classical ML μεθόδους.

## Φάση 2 (BERT)

In [20]:
MODEL_NAME = 'distilbert-base-uncased'

print(f'Loading tokenizer: {MODEL_NAME}')
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

print(f'Loading model: {MODEL_NAME}')
model = AutoModel.from_pretrained(MODEL_NAME)
model.eval() #evaluation mode - we dont train the model

print('Model and tokenizer loaded')
print(f'Model parameters: {sum(p.numel() for p in model.parameters()):,}')

Loading tokenizer: distilbert-base-uncased
Loading model: distilbert-base-uncased


Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

DistilBertModel LOAD REPORT from: distilbert-base-uncased
Key                     | Status     |  | 
------------------------+------------+--+-
vocab_layer_norm.weight | UNEXPECTED |  | 
vocab_transform.bias    | UNEXPECTED |  | 
vocab_transform.weight  | UNEXPECTED |  | 
vocab_projector.bias    | UNEXPECTED |  | 
vocab_layer_norm.bias   | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Model and tokenizer loaded
Model parameters: 66,362,880


## Cell BERT-1: Φόρτωση DistilBERT

Φόρτωση προ-εκπαιδευμένου μοντέλου `distilbert-base-uncased`.

**Στρατηγική — Feature Extraction (όχι fine-tuning):**
Επειδή δεν υπάρχει GPU, χρησιμοποιούμε το DistilBERT
μόνο για εξαγωγή embeddings (768 αριθμοί ανά κείμενο).
Τα embeddings δίνονται στο LinearSVC για classification.

**Παράμετροι μοντέλου:** 66,362,880

In [ ]:
def get_bert_embeddings(texts, batch_size=32):
    """
    Παίρνει Bert embeddings για μια λίστα κειμένων

    Για κάθε κείμενο:
    1. Tokenization
    2. Περνάει από DistilBERT
    3.Παίρνει το [CLS] token embedding (768 αριθμοί)
    που αναπαριστά ολόκληρο το κείμενο

    Args:
        texts: λίστα κειμένων
        batch_size: πόσα κείμενα επεξεργάζεται μαζί
    Returns:
        numpy arrau shape: (n_texts, 768)
    """

    all_embeddings = []

    #επεξεργασία σε batches
    for i in range(0, len(texts), batch_size):
        batch = texts[i:i+batch_size]

        #Tokenization
        encoded = tokenizer(
            batch,
            padding = True,  #exoun ola ta keimena idio mikos
            truncation=True, #kovoume megala keimena
            max_length=128,  #max length of tokens
            return_tensors='pt'  #epistrofi PyTorch tensors
        )

        #Eksagogi embeddings xoris training
        with torch.no_grad():      #dont calculate gradients -> faster
            outputs = model(**encoded)

        #Pairnoume to [CLS] token (first token) - anaparista olokliro to keimeno
        cls_embeddings = outputs.last_hidden_state[:, 0, :].numpy()
        all_embeddings.append(cls_embeddings)

        #Progress
        print(f' Επεξεργασία: {min(i+batch_size, len(texts))}/{len(texts)}', end='\r')

    return np.vstack(all_embeddings)

#Use title+text[:550] as the best baseline
texts_train = (train['title'].fillna('') + '' + train['text'].fillna('').str[:550]).tolist()
texts_valid = (valid['title'].fillna('') + '' + valid['text'].fillna('').str[:550]).tolist()
texts_test =  (test['title'].fillna('') + '' + valid['text'].fillna('').str[:550]).tolist()

texts_train = [str(t) for t in texts_train]
texts_valid = [str(t) for t in texts_valid]
texts_test = [str(t) for t in texts_test]

print('Extracting embeddings - Train set..')
X_train_bert = get_bert_embeddings(texts_train)
print(f'\n Train embeddings: {X_train_bert.shape}')

print('Extracting embeddings - Validation set ...')
X_valid_bert = get_bert_embeddings(texts_valid)
print(f'\n Valid embeddings: {X_valid_bert.shape}')

print('Extracting embeddings - Test set...')
X_test_bert = get_bert_embeddings(texts_test)
print(f'\n Test embeddings: {X_test_bert.shape}')


Extracting embeddings - Train set..
 Επεξεργασία: 5082/5082
 Train embeddings: (5082, 768)
Extracting embeddings - Validation set ...
 Επεξεργασία: 565/565
 Valid embeddings: (565, 768)
Extracting embeddings - Test set...
 Επεξεργασία: 997/997
 Test embeddings: (997, 768)


## Cell BERT-2: Εξαγωγή BERT Embeddings

Κάθε κείμενο (title + text[:550]) περνά από το DistilBERT
και παίρνουμε το **[CLS] token embedding** (768 αριθμοί)
που αναπαριστά ολόκληρο το κείμενο.

**Γιατί [CLS] token:**
Το BERT προσθέτει ένα ειδικό [CLS] token στην αρχή κάθε
κειμένου. Το embedding του συνοψίζει ολόκληρο το κείμενο
και χρησιμοποιείται για classification.

**Shapes:**
- Train: (5082, 768)
- Valid: (565, 768)
- Test: (997, 768)

In [ ]:
#SVM for hazard with BERT embeddings
clf_bert_hazard = LinearSVC(
    C=1.0,
    class_weight = 'balanced',
    max_iter=2000,
    random_state=42
)

clf_bert_hazard.fit(X_train_bert, y_train_hazard)
print('BERT+SVM hazard trained')

#SVM for product eith BERT embeddings
clf_bert_product = LinearSVC(
    C=1.0,
    class_weight='balanced',
    max_iter=2000,
    random_state=42
)

clf_bert_product.fit(X_train_bert, y_train_product)
print('BERT+SVM Product trained')

pred_bert_hazard = clf_bert_hazard.predict(X_valid_bert)
pred_bert_product = clf_bert_product.predict(X_valid_bert)

print('\n=== Rating BERT+SVM ===\n')
score_bert = official_st1_score(
    valid['hazard-category'], pred_bert_hazard,
    valid['product-category'], pred_bert_product
)

print(f'\n=== Comparisson of all models ===')
print(f'TF-IDF + LogReg (title only): 0.6097')
print(f'TF-IDF + LogReg (title+text[:550]): 0.6712')
print(f'TF-IDF + SVM    (title+text[:550]): 0.7263')
print(f'BERT   + SVM    (title+text[:550]): {score_bert:.4f}')

BERT+SVM hazard trained
BERT+SVM Product trained

=== Rating BERT+SVM ===

macro-F1 Hazard:   0.5496
Correct hazard predictions:  469/565 (83.0%)
macro-F1 Product (given correct h): 0.5024
----------------
OFFICIAL ST1 SCORE:   0.5260

=== Comparisson of all models ===
TF-IDF + LogReg (title only): 0.6097
TF-IDF + LogReg (title+text[:550]): 0.6712
TF-IDF + SVM    (title+text[:550]): 0.7263
BERT   + SVM    (title+text[:550]): 0.5260


## Cell BERT-3: BERT + SVM (Feature Extraction)

Χρήση των BERT embeddings (768 διαστάσεων) ως features για LinearSVC.

**Αποτελέσματα:**
| Μοντέλο | Score |
|---|---|
| TF-IDF + LogReg (title only) | 0.6097 |
| TF-IDF + LogReg (title+text[:550]) | 0.6712 |
| TF-IDF + SVM (title+text[:550]) | 0.7263 |
| BERT + SVM (feature extraction) | 0.5260 |

**Γιατί το BERT+SVM απέδωσε χειρότερα:**
1. Το DistilBERT είναι εκπαιδευμένο σε γενικό κείμενο —
   όχι σε domain-specific food hazard κείμενα
2. Το feature extraction δεν προσαρμόζει το BERT στο task μας
3. Το TF-IDF μαθαίνει ακριβώς από τα δικά μας δεδομένα

**Επόμενο βήμα:** Fine-tuning του DistilBERT στα δικά μας
δεδομένα για να προσαρμοστεί στο task.

In [ ]:
le_hazard = LabelEncoder()
le_product = LabelEncoder()

y_train_hazard_enc = le_hazard.fit_transform(y_train_hazard)
y_valid_hazard_enc = le_hazard.transform(y_valid_hazard)
y_train_product_enc = le_product.fit_transform(y_train_product)
y_valid_product_enc = le_product.transform(y_valid_product)

print(f'Hazard classes: {len(le_hazard.classes_)}')
print(f'Product classes: {len(le_product.classes_)}')
print(f'\nHazard labels: {le_hazard.classes_}')

Hazard classes: 10
Product classes: 22

Hazard labels: ['allergens' 'biological' 'chemical' 'food additives and flavourings'
 'foreign bodies' 'fraud' 'migration' 'organoleptic aspects'
 'other hazard' 'packaging defect']


## Cell BERT-3: Label Encoding

Μετατροπή string labels σε αριθμούς για το PyTorch.

- `allergens` → 0, `biological` → 1, κλπ (αλφαβητικά)
- **Hazard classes:** 10
- **Product classes:** 22

In [ ]:
class FoodHazardDataset(Dataset):
    """
    PyTorch Dataset για τα δεδομένα μας.
    Το PyTorch χρειάζεται τα δεδομένα σε αυτή τη μορφή
    για να τα φορτώνει σε batches κατά την εκπαίδευση.
    """
    def __init__(self, texts, labels, tokenizer, max_length=128):
        self.texts     = texts
        self.labels    = labels
        self.tokenizer = tokenizer
        self.max_length = max_length
    
    def __len__(self):
        # Πόσα δείγματα έχουμε συνολικά
        return len(self.texts)
    
    def __getitem__(self, idx):
        # Επιστρέφει ένα δείγμα με index idx
        text = str(self.texts[idx])
        label = self.labels[idx]
        
        # Tokenization
        encoding = self.tokenizer(
            text,
            max_length=self.max_length,
            padding='max_length',
            truncation=True,
            return_tensors='pt'
        )
        
        return {
            'input_ids':      encoding['input_ids'].squeeze(),
            'attention_mask': encoding['attention_mask'].squeeze(),
            'label':          torch.tensor(label, dtype=torch.long)
        }


# Δημιουργία datasets
train_texts = (train['title'].fillna('') + ' ' + train['text'].fillna('').str[:550]).tolist()
valid_texts = (valid['title'].fillna('') + ' ' + valid['text'].fillna('').str[:550]).tolist()
test_texts  = (test['title'].fillna('')  + ' ' + test['text'].fillna('').str[:550]).tolist()

# Hazard datasets
train_dataset_hazard = FoodHazardDataset(train_texts, y_train_hazard_enc, tokenizer)
valid_dataset_hazard = FoodHazardDataset(valid_texts, y_valid_hazard_enc, tokenizer)

# DataLoaders — φορτώνουν τα δεδομένα σε batches
train_loader_hazard = DataLoader(train_dataset_hazard, batch_size=16, shuffle=True)
valid_loader_hazard = DataLoader(valid_dataset_hazard, batch_size=16, shuffle=False)

print(f'Train batches: {len(train_loader_hazard)}')
print(f'Valid batches: {len(valid_loader_hazard)}')
print('Datasets έτοιμα')

Train batches: 318
Valid batches: 36
Datasets έτοιμα!


## Cell BERT-4: Dataset & DataLoader

Δημιουργία PyTorch Dataset και DataLoader.

- **Dataset**: πώς να διαβάζει το PyTorch κάθε δείγμα
- **DataLoader**: φορτώνει δεδομένα σε batches των 16
- **shuffle=True** στο train για καλύτερη εκπαίδευση
- Train batches: 318 | Valid batches: 36

In [ ]:
# Φόρτωση DistilBERT για classification
bert_hazard_model = AutoModelForSequenceClassification.from_pretrained(
    'distilbert-base-uncased',
    num_labels=len(le_hazard.classes_)  # 10 κλάσεις
)
bert_hazard_model = bert_hazard_model.to('cpu')

# Optimizer
optimizer = AdamW(bert_hazard_model.parameters(), lr=2e-5)

# Training function
def train_epoch(model, loader, optimizer):
    model.train()
    total_loss = 0
    
    for batch_idx, batch in enumerate(loader):
        input_ids      = batch['input_ids'].to('cpu')
        attention_mask = batch['attention_mask'].to('cpu')
        labels         = batch['label'].to('cpu')
        
        optimizer.zero_grad()           # μηδενίζουμε τα gradients
        outputs = model(               
            input_ids=input_ids,
            attention_mask=attention_mask,
            labels=labels
        )
        
        loss = outputs.loss            # υπολογισμός loss
        loss.backward()                # backpropagation
        optimizer.step()               # ενημέρωση βαρών
        total_loss += loss.item()
        
        if batch_idx % 50 == 0:
            print(f'  Batch {batch_idx}/{len(loader)} — Loss: {loss.item():.4f}', end='\r')
    
    return total_loss / len(loader)

# Evaluation function
def evaluate(model, loader):
    model.eval()
    all_preds = []
    
    with torch.no_grad():
        for batch in loader:
            input_ids      = batch['input_ids'].to('cpu')
            attention_mask = batch['attention_mask'].to('cpu')
            
            outputs = model(input_ids=input_ids, attention_mask=attention_mask)
            preds   = outputs.logits.argmax(dim=1).cpu().numpy()
            all_preds.extend(preds)
    
    return np.array(all_preds)

# Εκπαίδευση για 1 epoch
print('=== FINE-TUNING DISTILBERT — HAZARD ===')
train_loss = train_epoch(bert_hazard_model, train_loader_hazard, optimizer)
print(f'\nEpoch ολοκληρώθηκε! — Μέσο Loss: {train_loss:.4f}')

preds_enc = evaluate(bert_hazard_model, valid_loader_hazard)
preds_hazard_bert_ft = le_hazard.inverse_transform(preds_enc)

f1_hazard_bert = f1_score(y_valid_hazard, preds_hazard_bert_ft, average='macro', zero_division=0)
print(f'macro-F1 Hazard (fine-tuned BERT): {f1_hazard_bert:.4f}')
print(f'macro-F1 Hazard (TF-IDF + SVM):    0.7904')

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_layer_norm.weight | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
classifier.weight       | MISSING    | 
classifier.bias         | MISSING    | 
pre_classifier.weight   | MISSING    | 
pre_classifier.bias     | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


=== FINE-TUNING DISTILBERT — HAZARD ===
⚠️  Θα πάρει 20-40 λεπτά σε CPU — άφησέ το να τρέξει!

  Batch 300/318 — Loss: 0.7192
✅ Epoch ολοκληρώθηκε! — Μέσο Loss: 0.7495
macro-F1 Hazard (fine-tuned BERT): 0.5309
macro-F1 Hazard (TF-IDF + SVM):    0.7904


## Cell BERT-5: Fine-tuning DistilBERT — Hazard

Fine-tuning του DistilBERT για το hazard classification.

**Παράμετροι:**
- `lr=2e-5`: μικρό learning rate για BERT
- `batch_size=16`: 16 δείγματα ανά batch
- `epochs=1`: 1 epoch λόγω CPU

**Αποτέλεσμα 1 epoch:**
- Loss: 0.7495
- macro-F1 Hazard: 0.5309 (vs TF-IDF+SVM: 0.7904)

**Γιατί το αφήνουμε προσωρινά:**
Το fine-tuning σε CPU απαιτεί 20-40 λεπτά ανά epoch και
χρειάζεται 3-5 epochs για να συγκλίνει (2-3 ώρες συνολικά).
Επιλέγουμε να εξερευνήσουμε άλλες στρατηγικές πρώτα και
να επανέλθουμε αν υπάρχει χρόνος. Το καλύτερο μοντέλο
μας παραμένει το TF-IDF + SVM με score 0.72822 στο Kaggle.

In [ ]:
# Tuning της παραμέτρου C για το SVM
C_values = [0.01, 0.1, 0.5, 1.0, 5.0, 10.0]
results = []

for C in C_values:
    # Hazard classifier
    clf_h = LinearSVC(C=C, class_weight='balanced', max_iter=2000, random_state=42)
    clf_h.fit(X_train_best, y_train_hazard)
    
    # Product classifier
    clf_p = LinearSVC(C=C, class_weight='balanced', max_iter=2000, random_state=42)
    clf_p.fit(X_train_best, y_train_product)
    
    # Αξιολόγηση
    pred_h = clf_h.predict(X_valid_best)
    pred_p = clf_p.predict(X_valid_best)
    
    score = official_st1_score(
        valid['hazard-category'], pred_h,
        valid['product-category'], pred_p,
        verbose=False  # δεν εκτυπώνουμε αναλυτικά για κάθε C
    )
    
    results.append({'C': C, 'score': score})
    print(f'C={C:.2f} → Score: {score:.4f}')

# Εύρεση βέλτιστου C
best = max(results, key=lambda x: x['score'])
print(f'\nΒέλτιστο C: {best["C"]} με score: {best["score"]:.4f}')
print(f'Προηγούμενο best (C=1.0): 0.7263')

C=0.01 → Score: 0.5372
C=0.10 → Score: 0.6298
C=0.50 → Score: 0.7388
C=1.00 → Score: 0.7263
C=5.00 → Score: 0.7234
C=10.00 → Score: 0.7130

✅ Βέλτιστο C: 0.5 με score: 0.7388
Προηγούμενο best (C=1.0): 0.7263


## Cell Tuning-1: Hyperparameter Tuning — C στο SVM

Δοκιμή διαφορετικών τιμών C για το LinearSVC.

| C | Score |
|---|---|
| 0.01 | 0.5372 |
| 0.1 | 6298 |
| 0.5 | **0.7388** ✅ |
| 1.0 | 0.7263 |
| 5.0 | 0.7234 |
| 10.0 | 0.7130 |

**Βέλτιστο C: 0.5 με score 0.7388**

Μικρότερο C σημαίνει πιο "συντηρητικό" μοντέλο που
γενικεύει καλύτερα σε νέα δεδομένα. Το C=0.5 αποδίδει
καλύτερα από το C=1.0 (+0.0125) γιατί αποφεύγει
ελαφρύ overfitting στο training set.

In [ ]:
# Tuning max_features στο TF-IDF
feature_values = [5000, 10000, 20000, 50000, 100000]
results_features = []

for max_feat in feature_values:
    # Νέο TF-IDF
    tfidf_tune = TfidfVectorizer(
        max_features=max_feat,
        ngram_range=(1, 2),
        sublinear_tf=True,
        stop_words='english'
    )
    
    X_tr = tfidf_tune.fit_transform(train['combined'])
    X_vl = tfidf_tune.transform(valid['combined'])
    
    # SVM με βέλτιστο C=0.5
    clf_h = LinearSVC(C=0.5, class_weight='balanced', max_iter=2000, random_state=42)
    clf_h.fit(X_tr, y_train_hazard)
    
    clf_p = LinearSVC(C=0.5, class_weight='balanced', max_iter=2000, random_state=42)
    clf_p.fit(X_tr, y_train_product)
    
    pred_h = clf_h.predict(X_vl)
    pred_p = clf_p.predict(X_vl)
    
    score = official_st1_score(
        valid['hazard-category'], pred_h,
        valid['product-category'], pred_p,
        verbose=False
    )
    
    results_features.append({'max_features': max_feat, 'score': score})
    print(f'max_features={max_feat} → Score: {score:.4f}')

best_feat = max(results_features, key=lambda x: x['score'])
print(f'\nΒέλτιστο max_features: {best_feat["max_features"]} με score: {best_feat["score"]:.4f}')
print(f'Προηγούμενο best (10000): 0.7388')

max_features=5000 → Score: 0.6812
max_features=10000 → Score: 0.7388
max_features=20000 → Score: 0.7166
max_features=50000 → Score: 0.7436
max_features=100000 → Score: 0.7353

✅ Βέλτιστο max_features: 50000 με score: 0.7436
Προηγούμενο best (10000): 0.7388


## Cell Tuning-2: Hyperparameter Tuning — max_features TF-IDF

Δοκιμή διαφορετικών τιμών max_features με C=0.5.

| max_features | Score |
|---|---|
| 5000 | 0.6812 |
| 10000 | 0.7388 |
| 20000 | 0.7166 |
| **50000** | **0.7436** ✅ |
| 100000 | 0.7353 |

**Βέλτιστο: max_features=50000**
Μετά τα 50000 features εισάγεται θόρυβος που μειώνει
την απόδοση — παρόμοια συμπεριφορά με το text length.

In [ ]:
# Εκπαίδευση με βέλτιστες παραμέτρους: C=0.5, max_features=50000
tfidf_final = TfidfVectorizer(
    max_features=50000,
    ngram_range=(1, 2),
    sublinear_tf=True,
    stop_words='english'
)

X_tr_final = tfidf_final.fit_transform(train['combined'])
X_vl_final = tfidf_final.transform(valid['combined'])
X_te_final = tfidf_final.transform(test['combined'])

clf_h_final = LinearSVC(C=0.5, class_weight='balanced', max_iter=2000, random_state=42)
clf_h_final.fit(X_tr_final, y_train_hazard)

clf_p_final = LinearSVC(C=0.5, class_weight='balanced', max_iter=2000, random_state=42)
clf_p_final.fit(X_tr_final, y_train_product)

# Επαλήθευση στο validation
pred_h_final = clf_h_final.predict(X_vl_final)
pred_p_final = clf_p_final.predict(X_vl_final)

print('=== ΤΕΛΙΚΗ ΑΞΙΟΛΟΓΗΣΗ ===\n')
official_st1_score(
    valid['hazard-category'], pred_h_final,
    valid['product-category'], pred_p_final
)

# Παραγωγή submission
pred_h_test = clf_h_final.predict(X_te_final)
pred_p_test = clf_p_final.predict(X_te_final)

submission_final = pd.DataFrame({
    'id': test['id'],
    'hazard-category': pred_h_test,
    'product-category': pred_p_test
})

submission_final.to_csv('submission_tuned.csv', index=False)
print('\nSubmission file saved: submission_tuned.csv')
print(f'Αριθμός predictions: {len(submission_final)}')

=== ΤΕΛΙΚΗ ΑΞΙΟΛΟΓΗΣΗ ===

macro-F1 Hazard:   0.8040
Correct hazard predictions:  526/565 (93.1%)
macro-F1 Product (given correct h): 0.6833
----------------
OFFICIAL ST1 SCORE:   0.7436

Submission file saved: submission_tuned.csv
Αριθμός predictions: 997


## Cell Tuning-3: Τελικό Submission με Βέλτιστες Παραμέτρους

Εκπαίδευση με τις βέλτιστες παραμέτρους που βρέθηκαν:
- `C=0.5`
- `max_features=50000`
- `text[:550]`

**Kaggle Score: 0.74078**

| Submission | Kaggle Score |
|---|---|
| Baseline (title only, LogReg) | 0.57609 |
| LogReg (title + text[:550]) | 0.65231 |
| SVM (title + text[:550]) | 0.72822 |
| **SVM tuned (C=0.5, 50k features)** | **0.74078** |